In [90]:
import pandas as pd

In [91]:
df = pd.read_csv(
    "/home/chintan/rag_project/data/house_prices.csv",
    nrows=1000
)

In [92]:
def price_to_value(values):
    try:
        value = values.lower()
        if "cr" in  value:
            v = value.replace("cr","").strip()
            return float(v)*100
        v = value.replace("lac","").strip()
        if "lac" in  value:
            return float(value.replace("lac","").strip())
        return None
    except:
        return None

def car_parking_encoding(value):
    if type(value) == str and value and ("open" in value.lower() or "covered" in value.lower()):
        return 1
    return 0

48

def intTryParse(value):
    try:
        return int(value)
    except ValueError:
        return 0
    
def balcony_encoding(value):
    if type(value) != str:
        return 0
    # print(f"{value} {type(value)} {value} Before")
    if (type(value) == float or type(value) == int):
        return value
    return intTryParse(value)

def furnishing_encoding(value):
    if type(value) != str or "unfurnished" in value.lower():
        return 0
    if "semi-furnished" in value.lower() :
        return 0.5
    if "furnished" in value.lower() :
        return 1
    return 0


def overlooking_encoding(value):
    if type(value) != str:
        return None
    l = []
    if "road" in value.lower():
        l.append("road")
    if "garden" in value.lower() or "park" in value.lower():
            l.append("garden")
    if "pool" in value.lower():
            l.append("pool")   
    return ",".join(l)

In [93]:
df = df.drop(columns=["Price (in rupees)","Description","Title","Transaction","facing","Society", "Super Area","Plot Area","Dimensions", "Status", "location"])
df = df.rename(columns={'Amount(in rupees)': 'amount', 'Car Parking': 'car_parking','Carpet Area': 'area'})
df["amount"] = df["amount"].apply(price_to_value)
df["car_parking"] = df["car_parking"].apply(car_parking_encoding)
df["Balcony"] = df["Balcony"].apply(balcony_encoding)
df["Furnishing"] = df["Furnishing"].apply(furnishing_encoding)
df["Bathroom"] = df["Bathroom"].apply(intTryParse)
df["overlooking"] = df["overlooking"].apply(overlooking_encoding)
df["area"] = df["area"].apply(lambda value: intTryParse(value.split(" ")[0]) if value != None and type(value) == str  else None)
df["Floor"] = df["Floor"].apply(lambda value: intTryParse(value.split(" ")[0]) if value != None and type(value) == str  else None)
df = df.dropna(subset=["amount"])
df = df.dropna(subset=["area"])
df = df.dropna(subset=["Ownership"])
df = df.dropna(subset=["Floor"])
df = pd.get_dummies(df, columns=["overlooking","Ownership"], dtype=int)
df = df.drop(df[df["area"] > 3000].index)


In [ ]:
df[""]

### Train and Test Split

In [94]:
##
df_train = df.sample(frac=0.8,random_state=42)
df_test = df.drop(df_train.index)

### Train and Test X and Y values

In [95]:
X_train = df_train.drop("amount",axis=1)
Y_train = df_train["amount"]
X_test = df_test.drop("amount",axis=1)
Y_test = df_test["amount"]

In [96]:
df.head(n=5)
# X_test

,Index,amount,area,Floor,Furnishing,Bathroom,Balcony,car_parking,overlooking_garden,"overlooking_garden,pool",overlooking_pool,overlooking_road,"overlooking_road,garden","overlooking_road,garden,pool",Ownership_Co-operative Society,Ownership_Freehold,Ownership_Leasehold,Ownership_Power Of Attorney
1,1,98.0,473.0,3.0,0.5,2,0,1,1,0,0,0,0,0,0,1,0,0
2,2,140.0,779.0,10.0,0.0,2,0,1,1,0,0,0,0,0,0,1,0,0
4,4,160.0,635.0,20.0,0.0,2,0,1,0,0,0,0,1,0,1,0,0,0
9,9,160.0,900.0,3.0,0.0,3,1,1,1,0,0,0,0,0,0,1,0,0
10,10,140.0,950.0,6.0,0.5,2,0,1,0,0,0,1,0,0,1,0,0,0
